# 10 · Leave-One-Dataset-Out — Mendeley Held Out (3 seeds)

Trains the full method (configuration E, domain-adversarial) on OAI, NHANES III, and MRKR, and evaluates on the held-out Mendeley set across three random seeds.

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import sys, importlib
sys.path.insert(0,'/content/drive/MyDrive/Master Thesis/scope3')
import config; importlib.reload(config)
import pandas as pd, numpy as np
from pathlib import Path
import torch
if 'training_lib' in sys.modules: importlib.reload(sys.modules['training_lib'])
import training_lib as T
if not torch.cuda.is_available(): raise RuntimeError('No GPU. Runtime -> Change runtime type -> GPU.')
print('GPU:', torch.cuda.get_device_name(0))
manifest = T.prepare_local_data()
print(f'Manifest: {len(manifest):,} rows')

Mounted at /content/drive
GPU: NVIDIA A100-SXM4-40GB
Copied array in 43s
Loaded array (61558, 224, 224) in 1s
Manifest: 61,558 rows


In [2]:
def make_splits(manifest, test_dataset, seed=0):
    pool=manifest[manifest['dataset']!=test_dataset].copy()
    test=manifest[manifest['dataset']==test_dataset].copy()
    if 'split' in pool.columns and pool['split'].isin(['train','val']).any():
        tr=pool[pool['split'].isin(['train','TRAIN'])]; va=pool[pool['split'].isin(['val','VAL','validation'])]
        if len(va)==0: va=pool.sample(frac=0.15,random_state=seed); tr=pool.drop(va.index)
    else:
        va=pool.sample(frac=0.15,random_state=seed); tr=pool.drop(va.index)
    return tr.reset_index(drop=True), va.reset_index(drop=True), test.reset_index(drop=True)
print('Split helper ready.')

Split helper ready.


## Execution

In [3]:
fk='fold1_test_mendeley'; test_ds=config.LODO_FOLDS[fk]; mech=dict(config.FULL_METHOD)
try:
    _sw=pd.read_csv(str(config.RESULTS_DIR/'grl_lambda_sweep.csv'))
    _best=float(_sw.loc[_sw['external_acc5'].idxmax(),'grl_lambda_max'])
    mech['grl_lambda_max']=_best; print(f'Using swept GRL lambda: {_best}')
except Exception: print('No sweep file; using config GRL_LAMBDA_MAX')
results=[]
for seed in config.SEEDS:
    run=f'{fk}_seed{seed}'
    tr,va,te=make_splits(manifest,test_ds,seed=seed)
    print(f'--- {run} (train={len(tr):,} val={len(va):,} test={len(te):,}) ---',flush=True)
    r=T.run_training(run,tr,va,te,mech,seed,config.CKPT_DIR,config.RESULTS_DIR,log_fn=print)
    results.append(r)
df=pd.DataFrame(results)
print(f'External acc5: {df["external_acc5"].mean():.4f} +/- {df["external_acc5"].std():.4f}')
print(f'External QWK : {df["external_qwk"].mean():.4f}')
print(f'Gap          : {df["gap"].mean():.4f}')
df.to_csv(str(config.RESULTS_DIR/f'{fk}_results.csv'),index=False)

Using swept GRL lambda: 0.5
--- fold1_test_mendeley_seed0 (train=45,304 val=7,995 test=8,259) ---
Downloading: "https://download.pytorch.org/models/convnext_large-ea097f82.pth" to /root/.cache/torch/hub/checkpoints/convnext_large-ea097f82.pth


100%|██████████| 755M/755M [00:03<00:00, 243MB/s]


  [fold1_test_mendeley_seed0] resume ep3
  [fold1_test_mendeley_seed0] ep4/18 loss=2.289 tr=0.560 val=0.505 gap=+0.055 qwk=0.554 grl=0.34 (127s)
  [fold1_test_mendeley_seed0] ep5/18 loss=2.180 tr=0.583 val=0.514 gap=+0.068 qwk=0.564 grl=0.40 (107s)
  [fold1_test_mendeley_seed0] ep6/18 loss=2.154 tr=0.578 val=0.517 gap=+0.061 qwk=0.564 grl=0.44 (106s)
  [fold1_test_mendeley_seed0] ep7/18 loss=2.042 tr=0.578 val=0.525 gap=+0.053 qwk=0.569 grl=0.47 (107s)
  [fold1_test_mendeley_seed0] ep8/18 loss=1.964 tr=0.580 val=0.511 gap=+0.068 qwk=0.554 grl=0.48 (107s)
  [fold1_test_mendeley_seed0] ep9/18 loss=1.906 tr=0.580 val=0.533 gap=+0.047 qwk=0.581 grl=0.49 (107s)
  [fold1_test_mendeley_seed0] ep10/18 loss=1.842 tr=0.583 val=0.535 gap=+0.049 qwk=0.579 grl=0.49 (106s)
  [fold1_test_mendeley_seed0] ep11/18 loss=1.793 tr=0.579 val=0.533 gap=+0.046 qwk=0.581 grl=0.50 (107s)
  [fold1_test_mendeley_seed0] ep12/18 loss=1.742 tr=0.582 val=0.534 gap=+0.048 qwk=0.585 grl=0.50 (107s)
  [fold1_test_mendel